# How do you know your agent works?
### Evaluating Earth Observation agents: benchmarks, metrics, and a reusable harness

Hands-on session, BIFOLD Agentic AI for Earth Observation workshop. Pi School and ESA Φ-lab, EVE project.

Building an agent is straightforward. Showing that it works reliably is harder.

### What changes compared with evaluating an LLM

Scoring a plain LLM is mostly one question: *is the answer right?* The model gets a prompt and returns text, so you compare that text with a reference (accuracy, F1, BLEU, a judge). An **agent** is different:

- **It acts, not only answers.** It plans, calls tools (raster readers, index calculators, GIS operations, map APIs, code), reads the results, and decides what to do next. The answer is the last step of a longer process.
- **A right answer can hide a bad process.** The agent can guess, take a lucky shortcut, or waste ten calls. A wrong answer can come from one bad step in an otherwise sound run. The final answer alone cannot tell these apart.
- **The path is part of the result.** Which tools it chose, in which order, with which arguments, how many calls, and how it recovered from errors are all measurable, but only if the run is recorded as an **execution trace**.
- **Runs are not repeatable by default.** The same agent can take a different path on the same question, so you need fixed tasks, fixed tools, recorded configurations, and often several runs.
- **The environment is part of the system.** Tool schemas, data, the model endpoint, and timeouts change the score, so a number is only meaningful with the setup that produced it.

That is why agent evaluation has two levels: **answer-level** metrics (did it get the result?) and **trajectory-level** metrics (did it get there the right way?).

### What you will do

- survey existing EO-agent benchmarks and the metrics each one reports,
- see how the `agentic_evals` harness runs a benchmark: task, agent, tools, scorers, logs,
- assemble an EO agent, run it on Earth-Bench, and compute accuracy, Tool-Any-Order, Tool-In-Order, Tool-Exact-Match and Parameter Accuracy,
- add an LLM judge, and see where it agrees and disagrees with the metrics,
- diagnose failures from execution traces,
- put your own agent on a leaderboard next to the baseline, on identical tasks.

The model under test is EVE, served through an OpenAI-compatible endpoint. Any other endpoint works the same way.

## 1. Setup

This folder is a closed environment. `setup.sh` created `.venv` and `.env` here. Select the kernel **agentic-evals workshop** and fill in `EVE_API_KEY`, `EVE_BASE_URL`, and `MODEL_NAME` in `.env` before continuing.

The next cell points data and run output at this folder, then loads `.env`. If you restart the kernel, run it again.

In [ ]:
from pathlib import Path
import os, sys

root = Path.cwd()
if not (root / "workshop.py").is_file():
    candidate = root / "sagemaker"
    if (candidate / "workshop.py").is_file():
        root = candidate
    else:
        raise RuntimeError("Run this notebook with the working directory set to the sagemaker folder.")
os.chdir(root)
sys.path.insert(0, str(root))
sys.modules.pop("workshop", None)   # re-running this cell picks up edits to workshop.py
if Path(sys.prefix).resolve() != (root / ".venv").resolve():
    raise RuntimeError(f"Wrong kernel ({sys.executable}). Select the kernel 'agentic-evals workshop'.")

from dotenv import load_dotenv
load_dotenv(root / ".env")
if not os.environ.get("EVE_BASE_URL", "").strip() and os.environ.get("EVE_API_BASE_URL", "").strip():
    os.environ["EVE_BASE_URL"] = os.environ["EVE_API_BASE_URL"].strip()

os.environ["AGENTIC_EVALS_DATA"] = str((root / "data").resolve())
os.environ["AGENTIC_EVALS_RUNS_DIR"] = str((root / "runs").resolve())
# Tool servers write their banners, warnings and tracebacks to runs/.../kit_stderr.log, not here.
os.environ["AGENTIC_EVALS_KIT_STDERR"] = "file"
# With W&B off, turn the inspect-wandb hooks off too. Otherwise they log a warning on every event.
if os.environ.get("WANDB_MODE", "").strip() == "disabled":
    os.environ["INSPECT_WANDB_MODELS_ENABLED"] = "false"
    os.environ["INSPECT_WANDB_WEAVE_ENABLED"] = "false"

assert sys.version_info >= (3, 12), f"Python 3.12+ needed, found {sys.version.split()[0]}"
if "agentic_evals.paths" in sys.modules:
    from agentic_evals.paths import DATA_DIR
    expected = (root / "data").resolve()
    if DATA_DIR != expected:
        raise RuntimeError("Restart the kernel so AGENTIC_EVALS_DATA is picked up.")

REPO = root.parent   # the agentic-evals checkout this folder belongs to
print("folder", root)
print("python", sys.executable)

In [ ]:
import pandas as pd
from workshop import (
    WORKSHOP_QUESTIONS, check_environment, download_earth_bench, judge, leaderboard, results_df, run,
)
from agentic_evals.agents.langgraph_react import langgraph_agent

pd.set_option("display.precision", 2)
MODEL_NAME = os.environ["MODEL_NAME"].strip()
MODEL = f"openai-api/eve/{MODEL_NAME}"
print("model under test", MODEL)
print("workshop questions", list(WORKSHOP_QUESTIONS))
check_environment()

## 2. Metrics and benchmarks for EO agents

### 2.1 The usual metrics for agents

Agent metrics fall into three families. Most benchmarks report one or two of them.

**Answer-level** (scores the final result)
- *Accuracy / exact match*: the chosen option, number or label equals the gold one.
- *Pass@k*: the share of tasks solved in at least one of k attempts, for agents whose runs vary.
- *LLM-as-a-judge*: a model decides whether a free-text answer or a whole solution matches the reference (match, partial, no match).

**Trajectory-level** (scores the path against an expert reference)
- *Tool-Any-Order*: did the agent use the tools the expert used?
- *Tool-In-Order*: did it use them in the same order, extra calls allowed?
- *Tool-Exact-Match*: is the tool sequence identical, with no extra calls?
- *Parameter accuracy*: are the arguments of each call right?
- *Step accuracy*: instruction, tool, argument and summary accuracy checked per step (ThinkGeo).

**Cost and reliability** (scores how the run went)
- *Efficiency / length*: number of calls compared with the expert's.
- *Error rate*: share of tool calls or generated code that fail to run.
- *Termination*: did the run finish, or hit a call or step limit?

### 2.2 Benchmarks

| Benchmark | Focus | Metrics | In harness |
| --- | --- | --- | --- |
| **Earth-Bench** (Feng et al., ICLR 2026) | Raster analysis with 104 fixed tools | Accuracy; TAO, TIO, TEM, ParamAcc, efficiency | yes |
| **GeoBenchX** (Krechetova & Kochedykov, SIGSPATIAL 2025) | GIS workflows, including tasks that must be rejected | LLM-judge match / partial / no match; length difference | yes |
| **EO-Gym** (arXiv:2605.01250) | Multimodal tasks with imagery, tool server | Pass@k with LLM judge; binary tool metrics; parameters | in progress |
| **OpenEarthAgent** | Tool-based EO agents | to be added | in progress |
| **ThinkGeo** (Shabbir et al., 2025) | Optical and SAR remote sensing, 14 tools | Judge-based answer accuracy; instruction, tool, argument, summary accuracy | no |
| **UnivEARTH** (Kao et al., 2025) | Yes/no questions solved with Earth Engine code | Accuracy; code failure rate | no |
| **GeoCode-Bench** (Hou et al., 2024) | Geospatial code generation | Answer accuracy | no |
| **MapEval-API** (Dihan et al., ICML 2025) | Map-API questions: places, routes, distances | Accuracy | no |

Three things to take from this table:

1. **Every benchmark brings its own metrics.** This harness implements each benchmark's published metrics and does not mix them into one shared score.
2. **The same name can mean different things.** "Tool exact match" is a fraction in Earth-Bench and a 0/1 flag in EO-Gym (section 3).
3. **LLM judges are part of some protocols.** For GeoBenchX, EO-Gym and ThinkGeo the official answer score comes from a judge model, so the judge must be reported with the number.

To pick a benchmark, start from the capability you care about: raster analysis with fixed tools (Earth-Bench), GIS workflows and knowing when to refuse (GeoBenchX), visual grounding with imagery (EO-Gym, ThinkGeo), code against a real data platform (UnivEARTH, GeoCode-Bench), or map services (MapEval-API).

## 3. Reading Earth-Bench's metrics

We start with **Earth-Bench** ([Feng et al., ICLR 2026](https://arxiv.org/abs/2509.23141)), because it reports both levels and ships an expert trajectory for most tasks.

- **What it is:** a benchmark of 248 multiple-choice Earth Observation questions that an agent answers by calling tools on real satellite data.
- **Data:** three modalities: Spectrum (questions 1 to 100), Products (101 to 188) and RGB (189 to 248). We use Spectrum.
- **Tools:** 104 tools in five MCP kits (Analysis, Index, Inversion, Perception, Statistics): file listing, land surface temperature, vegetation indices, thresholds, statistics.
- **Two regimes:** in **AP** (auto-planning) the agent chooses its own steps; in **IF** (instruction-following) the question lists them. Always report them separately. We use AP.
- **Gold standard:** a correct option, plus an expert list of tool calls for the task.
- **Metrics:** Appendix B.2 defines the six below. The harness computes them with `agentic_evals.tasks.earth_bench.metrics`, which follows upstream `evaluate/step_by_step.py` and is tested against the published score files.

Let \(t^*\) be the expert tool list with \(m\) steps and \(t\) the agent's list:

| Metric | Level | Meaning |
| --- | --- | --- |
| **Accuracy** | answer | The chosen option (A to D) equals the gold option. |
| **TAO** (Tool-Any-Order) | trajectory | Share of expert tools that the agent used at all. |
| **TIO** (Tool-In-Order) | trajectory | How many expert tools, from the start, appear in the agent's list in the same order. Extra calls in between are allowed. |
| **TEM** (Tool-Exact-Match) | trajectory | Longest common prefix of \(t\) and \(t^*\), divided by \(m\). No extra calls allowed. |
| **ParamAcc** | trajectory | Like TEM, but each call must also have exactly the expert's arguments. |
| **Efficiency** | cost | \(\lvert t \rvert / \lvert t^* \rvert\). A ratio, not a score: above 1 means more calls than the expert. |

A small example. The agent calls `get_filelist` twice, then finishes the task like the expert:

In [ ]:
from agentic_evals.tasks.earth_bench import metrics

def call(name, **kw):
    return {"name": name, "input": kw}

gold = [
    call("get_filelist", dir_path="benchmark/data/question11"),
    call("lst_multi_channel", band31_path="b31.tif", band32_path="b32.tif", output_path="lst.tif"),
    call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=310.0),
]
pred = [
    call("get_filelist", dir_path="benchmark/data/question11"),
    call("get_filelist", dir_path="benchmark/data/question11"),   # extra call
    call("lst_multi_channel", band31_path="b31.tif", band32_path="b32.tif", output_path="lst.tif"),
    call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=310.0),
]

print("TAO       ", metrics.tool_any_order(pred, gold))
print("TIO       ", metrics.tool_in_order(pred, gold))
print("TEM       ", round(metrics.tool_exact_match(pred, gold), 3))
print("ParamAcc  ", round(metrics.parameter_accuracy(pred, gold), 3))
print("Efficiency", round(metrics.efficiency(pred, gold), 3))
print("First divergence at step", metrics.first_divergence_step(pred, gold))

TAO and TIO forgive the extra call. TEM and ParamAcc do not, so one wasted call early in the run caps them at 1/3. The metrics answer different questions: *did it use the right tools*, *in the right order*, *without detours*, *with the right arguments*.

Now a wrong argument instead of an extra call. The agent picks a different LST threshold:

In [ ]:
pred_wrong_param = [gold[0], gold[1], call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=300.0)]

print("TAO, TIO, TEM:", metrics.tool_any_order(pred_wrong_param, gold),
      metrics.tool_in_order(pred_wrong_param, gold), metrics.tool_exact_match(pred_wrong_param, gold))
print("ParamAcc     :", round(metrics.parameter_accuracy(pred_wrong_param, gold), 3))

Only ParamAcc sees that error, and an answer-only score would see it only if the threshold changes the selected option.

### Same names, different definitions

Two other benchmarks in this harness report tool metrics too:

- **EO-Gym** (arXiv:2605.01250, port in progress; its metric functions are already in the harness) gives the agent imagery and a tool server. Each run is one *trajectory*; its tool metrics are 0/1 flags, and answers are summarised as Pass@k.
- **GeoBenchX** (Krechetova & Kochedykov, SIGSPATIAL 2025) has 202 GIS tasks with 24 tools, including tasks that should be rejected. Its official score is a judge comparing the whole solution with the reference, plus the length difference.

Each benchmark in this harness keeps its own metric module. Here is the same toy trajectory (`pred`, with the extra call) scored with **EO-Gym**'s and **GeoBenchX**'s published definitions:

In [ ]:
from agentic_evals.tasks.eo_gym import metrics as eo_gym
from agentic_evals.tasks.geobenchx import metrics as geobenchx

gold_names, pred_names = metrics.tool_names(gold), metrics.tool_names(pred)
pd.DataFrame(
    {
        "Earth-Bench (fraction)": [metrics.tool_any_order(pred, gold), metrics.tool_in_order(pred, gold),
                                   metrics.tool_exact_match(pred, gold)],
        "EO-Gym (0/1 per trajectory)": [int(eo_gym.tools_any_order(gold_names, pred_names)),
                                        int(eo_gym.tools_in_order(gold_names, pred_names)),
                                        int(gold_names == pred_names)],
    },
    index=["any order", "in order", "exact match"],
)

In [ ]:
# Length: Earth-Bench reports a ratio, GeoBenchX a signed relative difference.
print("Earth-Bench efficiency  |t| / |t*|          =", round(metrics.efficiency(pred, gold), 3))
print("GeoBenchX length_diff   (n - mean_ref)/mean_ref =", round(geobenchx.length_diff(len(pred), [gold]), 3))

# EO-Gym answers are judged per trajectory and summarised as Pass@k.
# 3 trajectories, 1 correct: unbiased Pass@1 and Pass@3 (Chen et al., 2021).
print("EO-Gym unbiased Pass@1 =", round(eo_gym.unbiased_pass_at_k(n=3, c=1, k=1), 3),
      "  Pass@3 =", eo_gym.unbiased_pass_at_k(n=3, c=1, k=3))

Same trajectory, different numbers: **never compare a "TEM" or an "efficiency" across benchmarks**. A leaderboard row only means something next to rows from the same benchmark, the same regime, and the same tasks.

### Answers that are not multiple choice: LLM-as-a-judge

Earth-Bench answers are option letters, so exact match is enough. When the answer is a GIS workflow or free text, benchmarks use a judge model. GeoBenchX's official score asks a judge to compare the candidate solution with the reference solutions and reply with a score in tags. This is the parser the harness uses for that reply:

In [ ]:
reply = (
    "<MATCH REASONING>The candidate loads the same dataset and filters the same countries, "
    "but colours the map by a different column.</MATCH REASONING>\n<MATCH SCORE>1</MATCH SCORE>"
)
print(geobenchx.parse_verdict(reply))   # 0 no match, 1 partial, 2 match
print(geobenchx.parse_verdict("Looks right to me!"))   # no tag: the sample is left unscored, not guessed

The judge model is part of that number. Swap the judge and the score can move, so the harness records the judge with every verdict and keeps each judge as its own result. We come back to judges in section 8.

## 4. How the harness runs an eval

`agentic_evals` is built on [Inspect AI](https://inspect.aisi.org.uk/). One evaluation is an Inspect **Task**:

```
dataset   questions, gold answers, gold trajectories          (tasks/<benchmark>/dataset.py)
setup     record the endpoint, start this sample's MCP tool servers
solver    your agent: a LangGraph graph run through agent_bridge() (agents/)
scorers   the benchmark's published metrics, plus optional extra judges (tasks/<benchmark>/scorers.py)
   |
   +--> .eval log: every message, tool call, score, and the model/endpoint identity
```

Let us build the Earth-Bench task object and look inside:

In [ ]:
from inspect_ai._util.registry import registry_info
from agentic_evals.tasks.earth_bench.task import earth_bench

task = earth_bench(regime="ap", modality="spectrum")
print("samples :", len(task.dataset), "(AP, Spectrum)")
print("setup   :", [s.__qualname__.split(".")[0] for s in task.setup])
print("solver  :", registry_info(task.solver).name, "(default; we pass our own below)")
print("scorers :", [registry_info(s).name for s in task.scorer])
print("metrics the paper reports:", task.metadata["published_metrics"])

The agent never sees benchmark code. It gets the sample's messages and whatever MCP tools the task published. That is why the same agent can run on Earth-Bench and GeoBenchX (and on EO-Gym once its port is done), and why the scorer, not the agent, decides what counts as correct.

A run is described in a YAML file in `configs/` (task, model, generation settings, which samples, epochs), so it can be repeated exactly. This is the CLI version of the baseline you will run in section 6:

In [ ]:
print((REPO / "configs" / "workshop_earth_bench_eve.yaml").read_text())

From a terminal at the repository root:

```bash
uv run inspect eval --run-config configs/workshop_earth_bench_eve.yaml   # run
uv run inspect view                                                     # browse transcripts, traces, scores
uv run inspect score logs/<run>.eval --scorer agentic_evals/earth_bench_scorer   # rescore offline
```

In this notebook we call the same thing from Python (`inspect_ai.eval`), because the agent graph is defined here. `workshop.run()` is a thin wrapper that fixes the task, the question set, and the timeouts.

## 5. Earth-Bench: questions, data, tools

### 5.1 Questions

`WORKSHOP_QUESTIONS` is our workshop subset of Earth-Bench:

- **10 Spectrum tasks** in the AP regime, all with a gold trajectory.
- **3 tool calls each**, except question 8, which has 4.
- **Different retrieval methods** (TVDI, LST, band ratio, TES, TTM, split-window, MODIS, ATI), so the agent cannot reuse one recipe.
- **Not an official split.** Earth-Bench does not define this subset; we picked it so the session stays short.

Each question is multiple choice and comes with its gold tool trajectory.

In [ ]:
from agentic_evals.tasks.earth_bench.dataset import load_samples

ap = {s.id: s for s in load_samples("ap", "spectrum") if s.id in WORKSHOP_QUESTIONS}
sample = ap[11]

system_prompt, question = sample.input.split("Question:", 1)
print("SYSTEM PROMPT (sent as part of the user message)", system_prompt.strip(), sep="\n")
print("\nQUESTION (AP), with the data folder appended and the choices", question.strip(), sep="\n")
print("\nGold answer:", sample.target)
print("\nGold trajectory:")
for i, c in enumerate(sample.metadata["gold_tool_calls"], 1):
    print(f"  {i}. {c['name']}({c['input']})")

In [ ]:
# The same question in the IF regime: the steps are written in the question.
IF = {s.id: s for s in load_samples("if", "spectrum") if s.id in WORKSHOP_QUESTIONS}
print(IF[11].input.split("Question:", 1)[1].strip())

### 5.2 Data

The rasters come from the public Hugging Face dataset `Sssunset/Earth-Bench` (the full benchmark is 16.6 GB). We download only the folders for our 10 questions, into this folder's `data/`.

In [ ]:
raster_root = download_earth_bench(WORKSHOP_QUESTIONS)
files = [p for p in raster_root.rglob("*") if p.is_file() and ".cache" not in p.parts]
print(f"{len(files)} files in {raster_root}  ({sum(p.stat().st_size for p in files) / 1e6:.1f} MB)")
print(sorted(p.name for p in raster_root.iterdir() if not p.name.startswith(".")))

### 5.3 Tools

The agent does not call Python functions directly. Earth-Bench's tools run as five [MCP](https://modelcontextprotocol.io) servers (stdio), one per kit. Any MCP client can use them, which is the same idea as EVE's tool registry.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from agentic_evals.tasks.earth_bench.tools import KITS, kit_servers

# Each kit starts its own server process. Its diagnostics go to runs/demo/kit_stderr.log.
client = MultiServerMCPClient(kit_servers(Path("runs/demo").resolve()))
for kit in KITS:
    kit_tools = await client.get_tools(server_name=kit)
    print(f"{kit:<11} {len(kit_tools):>3} tools   e.g. {', '.join(t.name for t in kit_tools[:4])}")

In [ ]:
# Call one tool by hand, the way the agent will.
tools = await client.get_tools()
by_name = {t.name: t for t in tools}
print(len(tools), "tools in total")
print(await by_name["get_filelist"].ainvoke({"dir_path": "benchmark/data/question11"}))

Every model call carries all these tool descriptions, which is a lot of tokens. Remember this when you design your own graph: a longer loop costs more.

## 6. Assemble an agent, run it, score it

### 6.1 The agent: ReAct

ReAct ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) alternates *reasoning*, *acting* (a tool call) and *observing* (the tool result) until the model answers.

The graph is defined in the next cell. `create_agent` is LangChain 1.x's ReAct graph. Two pieces of middleware sit on it: one repairs Mistral tool calls that arrive as text, and one moves images out of tool messages, because chat-completions APIs accept images only on user messages.

`langgraph_agent(react_graph)` is the harness around the graph. It starts the MCP servers, routes `llm` to your model, gives tool errors back to the model instead of crashing, and records every tool call as a trace. The scorer reads that trace. You do not edit the harness.

In [ ]:
from langchain.agents import create_agent
from agentic_evals.agents.langgraph_react import NormalizeMistralToolCalls, ToolImagesAsUserMessage

def react_graph(llm, tools):
    return create_agent(
        llm,
        tools=tools,
        middleware=[NormalizeMistralToolCalls(), ToolImagesAsUserMessage()],
        name="workshop_react",
    )

### 6.2 One question, with the raw Inspect call

`workshop.run()`, used later, is a thin wrapper around this call. `strict_tools=False` is required: some Earth-Bench tool schemas have untyped arguments that strict mode rejects.

In [ ]:
from inspect_ai import eval as inspect_eval
from workshop import check_budget

check_budget()   # one-token request: stops here if the model's token budget is spent

one = inspect_eval(
    earth_bench(regime="ap", modality="spectrum", solver=langgraph_agent(react_graph, name="workshop_react")),
    model=MODEL,
    model_args={"strict_tools": False},
    sample_id=[11],
    log_dir="logs/one",
    display="none",
    max_tokens=4096,          # cap each reply
    attempt_timeout=180,      # give up on a silent call after 3 minutes
    stream_idle_timeout=60,
    max_retries=6,            # back off and retry on HTTP 429 (the room shares one endpoint)
    max_connections=2,        # at most 2 model calls in flight from this notebook
)[0]
print("status:", one.status)

In [ ]:
import json
from agentic_evals.agents.trace_callback import read_trace

def show_trace(log, question):
    s = next(s for s in log.samples if int(s.id) == int(question))
    for e in read_trace(s.store):
        print(f"{e['step']:>2}. {e['name']}({json.dumps(e['input'])[:140]})")
        print("    ->", str(e.get("output"))[:120], "  [ERROR]" if e.get("error") else "")
    print("\nFinal message:", s.output.completion[-200:])

show_trace(one, 11)

This is the **execution trace**: every tool call with its arguments and output, in order. Everything that follows (trajectory metrics, the judge, your diagnosis) is computed from it.

### 6.3 The score of that sample

In [ ]:
score = one.samples[0].scores["earth_bench_scorer"]
md_ = score.metadata

print({k: round(v, 3) for k, v in score.value.items()})
print("\nextracted answer :", score.answer, "| gold:", one.samples[0].target)
print("gold tools       :", md_["gold_tools"])
print("predicted tools  :", md_["predicted_tools"])
print("first divergence :", md_["first_divergence_step"])
for step in md_["step_matches"]:
    print("  ", step)

`tool_errors` counts calls that raised an error or returned one (for example a made-up file path). `perception_failures` should stay at 0 here; if it does not, the setup is broken, not the model.

### 6.4 All 10 questions

This runs up to 4 samples in parallel and takes a few minutes. Each question costs about 120k tokens of your model budget, because every model call carries the descriptions of all 104 tools.

In [ ]:
baseline = run(
    langgraph_agent(react_graph, name="workshop_react"),
    model=MODEL,
    questions=WORKSHOP_QUESTIONS,
    log_dir="logs/baseline",
    max_samples=4,
)
df_base = results_df(baseline)
df_base[["accuracy", "tao", "tio", "tem", "param_acc", "efficiency", "tool_errors", "n_calls", "termination"]]

In [ ]:
COLS = ["accuracy", "tao", "tio", "tem", "param_acc", "efficiency", "tool_errors"]
print("Baseline, mean over", len(df_base), "questions (Earth-Bench AP, Spectrum)")
df_base[COLS].mean().round(3)

With 10 questions, one answer moves accuracy by 10 points. Treat differences of a few points as noise; the point of this session is the method, not the ranking.

Look at the rows where accuracy and the trajectory metrics disagree:

- **right answer, low TEM**: the agent found another route, or got lucky;
- **wrong answer, high TIO**: right tools, wrong arguments or a wrong reading of the result.

An answer-only score cannot tell these cases apart. That is the reason trajectory metrics exist.

In [ ]:
df_base.assign(
    right_answer=df_base.accuracy == 1,
    expert_path=df_base.tem == 1,
)[["accuracy", "tio", "tem", "param_acc", "right_answer", "expert_path"]]

## 7. Diagnose a failure from its trace

Pick the question where the agent's path was furthest from the expert's, and read its trace next to the gold trajectory.

In [ ]:
worst = df_base.sort_values(["tio", "accuracy"]).index[0]
print("Question", worst, df_base.loc[worst, ["accuracy", "tao", "tio", "tem", "param_acc"]].round(2).to_dict())
print("Gold:")
for i, c in enumerate(ap[worst].metadata["gold_tool_calls"], 1):
    print(f"  {i}. {c['name']}({json.dumps(c['input'])[:140]})")
print("\nAgent:")
show_trace(baseline, worst)

Questions to ask while you read:

- Did the agent choose a different but valid tool, or a wrong one?
- Did it get the arguments wrong (paths, thresholds, band order)?
- Did it hit a tool error and recover, or give up? The tool server's own traceback is in `runs/earth_bench/question<N>/<epoch-id>/kit_stderr.log`.
- Is the final answer right even though the trajectory differs from the expert's? What does that say about each metric?

The `.eval` log keeps the full transcript. `inspect view` (run from this folder: `.venv/bin/inspect view --log-dir logs`) shows every model message next to every tool call.

## 8. LLM-as-a-judge and its limits

Exact-match metrics are strict: a different but valid path scores low. An **LLM judge** reads the agent's trajectory and the reference trajectory and decides whether the agent's was accurate. The harness uses the reference-trajectory prompt from [agentevals](https://github.com/langchain-ai/agentevals) (`TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE`) and asks for a true/false verdict with reasoning (`agentic_evals.scorers.judges.trajectory_judge`, metric `trajectory_accuracy`).

For Earth-Bench this judge is **our addition**, not part of the paper's protocol. It is stored as a separate scorer and never mixed into Earth-Bench's numbers.

Because the log keeps the full trace, we do not rerun the agent: we **rescore the saved baseline log**.

In [ ]:
JUDGE = MODEL   # the judge can be a different model; always report which one you used

judged = judge(baseline, JUDGE)
df_j = results_df(judged)
df_j[["trajectory_judge_trajectory_accuracy", "accuracy", "tio", "tem", "param_acc"]]

In [ ]:
verdicts = {
    int(s.id): s.scores["trajectory_judge"].explanation
    for s in judged.samples if "trajectory_judge" in (s.scores or {})
}
disagree = df_j[(df_j.trajectory_judge_trajectory_accuracy == 1) != (df_j.tem == 1)]
print(f"{len(disagree)} of {len(df_j)} questions where the judge and Tool-Exact-Match disagree\n")
for q in disagree.index[:3]:
    print(f"Q{q}  judge={df_j.loc[q, 'trajectory_judge_trajectory_accuracy']}  TEM={df_j.loc[q, 'tem']:.2f}  accuracy={df_j.loc[q, 'accuracy']}")
    print("   ", verdicts.get(q, "")[:400], "\n")

Read a few of the judge's explanations against the traces. Then think about the limits:

- **The verdict depends on the judge model.** Scores from different judges are not comparable; report the judge with the number. Here the judge is the same model as the agent, which invites **self-preference**.
- **It sees truncated tool outputs** and may accept plausible but wrong parameters (try it on the threshold example in section 3).
- **A boolean hides partial credit.** TIO, TEM and ParamAcc do not.
- **It costs a model call per sample** and is not deterministic across providers.

When a judge *is* the benchmark's protocol (GeoBenchX, EO-Gym, ThinkGeo), use the judge the paper used, and keep each judge's verdict separate rather than majority-voting them.

## 9. Your agent on the leaderboard

Any LangGraph graph can be evaluated, as long as it follows this contract. `langgraph_agent(factory)` handles the rest (tools, model routing, trace, scoring).

- Your **factory** is a function `(llm, tools) -> compiled graph`.
- `llm` is already routed to your model. Call `llm.bind_tools(tools)` yourself if your nodes need tools.
- The graph state must have a `messages` key (use `MessagesState`). The first message already contains the system prompt and the question.
- The **last LLM call** must produce the final answer in the format `<Answer>B<Answer>`. A node that edits the answer after the last LLM call will not be scored.
- Tool calls are traced automatically, in any node or subgraph.

Here is a ReAct graph written by hand, node by node. It behaves like the baseline, without the two middleware pieces from section 6.

In [ ]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

def my_react_graph(llm, tools):
    llm_with_tools = llm.bind_tools(tools)

    async def agent_node(state: MessagesState):
        return {"messages": [await llm_with_tools.ainvoke(state["messages"])]}

    g = StateGraph(MessagesState)
    g.add_node("agent", agent_node)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)   # tool call -> "tools", else end
    g.add_edge("tools", "agent")
    return g.compile()

### 9.1 Exercise

Start from `my_react_graph`, change one thing, and see what the metrics say. Pick one idea, or bring your own:

1. **Plan first.** Add a node before the loop that asks the LLM for a numbered plan using the tool names, then run ReAct with the plan in context.
2. **Check the format.** After the loop, add a node that asks the LLM to re-read its answer and output only `<Answer>X<Answer>`. (It must be an LLM call, see the contract above.)
3. **Limit retries.** Add a counter in the state and stop calling tools after N errors.
4. **Change the prompt.** Add a short system message with advice for EO tools (check file lists first, reuse returned paths).

Give your agent a name: it is the row label on the leaderboard. Test on 2 or 3 questions first, then run all 10 so the run counts.

In [ ]:
AGENT_NAME = "my_agent"   # change me: your team or idea, e.g. "plan_first_team3"

def my_graph(llm, tools):
    # TODO: start from my_react_graph and change it
    return my_react_graph(llm, tools)

mine = run(
    langgraph_agent(my_graph, recursion_limit=50, name=AGENT_NAME),
    model=MODEL,
    questions=WORKSHOP_QUESTIONS[:3],      # use WORKSHOP_QUESTIONS for the leaderboard run
    log_dir="logs/mine",
    max_samples=3,
)
df_mine = results_df(mine)
df_mine[COLS + ["n_calls"]]

In [ ]:
# Compare on the same questions only.
same = df_base.index.intersection(df_mine.index)
pd.concat(
    {"baseline": df_base.loc[same, COLS].mean(), AGENT_NAME: df_mine.loc[same, COLS].mean()}, axis=1
).round(3)

Did your change fix the failure you found in section 7, or move it somewhere else? Look at trajectories, not only accuracy.

### 9.2 The leaderboard

`leaderboard()` reads every finished run in `logs/` that covered **exactly** the ten workshop questions, one row per run. It does not invent a combined score: the columns are Earth-Bench's own metrics, sorted by accuracy, and judge verdicts (if a run has them) are separate columns. AP and IF are separate rows.

For a shared room leaderboard, everyone writes runs to one log directory (Inspect accepts `s3://` paths) and calls `leaderboard(<that directory>)`.

In [ ]:
leaderboard("logs")

## 10. Wrap-up

What you did:

1. Surveyed EO-agent benchmarks and saw that each one scores something different, with its own definitions.
2. Separated answer-only from trajectory evaluation, and computed Accuracy, TAO, TIO, TEM and ParamAcc.
3. Ran an agent through a harness that records every tool call, and read the trace of a failure.
4. Added an LLM judge by rescoring a saved log, and saw where it disagrees with the metrics.
5. Wrote your own graph and compared it with the baseline on identical tasks.

### Single agent vs multi-agent

*Coming soon.* The same harness will run a multi-agent system (an orchestrator with specialist agents) on the same ten questions, so it lands on the same leaderboard as a single agent. The trace already records which agent made each call.

### Next steps

- Run the other regime (`regime="if"`) and report AP and IF separately.
- Try more questions or other Earth-Bench modalities (`download_earth_bench` with a products question list needs about 12 GB).
- Run GeoBenchX through the same harness (see the repository README) and watch the metrics change meaning. EO-Gym and OpenEarthAgent are in progress.
- Write an MCP server for your own EO service and contribute it to the EVE tool registry.

### Troubleshooting

- `401` or `403` from the model: check `EVE_API_KEY` and `EVE_BASE_URL` in this folder's `.env`.
- `404 model not found`, or `check_environment` says the endpoint does not list the model: check `MODEL_NAME`. The check is `GET {EVE_BASE_URL}/models`, the same call the harness records on each sample.
- `BudgetExceeded`: your key's token budget is spent; the message says when it resets. Retrying will not help; ask the organizers for a key with budget. A full pass of this notebook uses about 1.8M tokens.
- `RateLimitError` or HTTP 429: the endpoint is busy. `run()` already backs off and retries; if it persists, lower `max_samples` and wait a minute.
- `Wrong kernel` in the setup cell, or `ModuleNotFoundError`: select the kernel **agentic-evals workshop** (top right in Jupyter and VS Code).
- Many `tool_errors`: read the trace with `show_trace`. Usually the model invented a file path.
- A run is very slow: lower `max_samples` or test on `WORKSHOP_QUESTIONS[:3]`.
- Your run is missing from the leaderboard: it must finish and cover all ten questions.
- Import errors after install: restart the kernel and run the setup cell again.

### References

- Feng et al. *Earth-Agent: Unlocking the Full Landscape of Earth Observation with Agents* (Earth-Bench). ICLR 2026. [arXiv:2509.23141](https://arxiv.org/abs/2509.23141)
- Krechetova & Kochedykov. *GeoBenchX: Benchmarking LLMs in Agent Solving Multistep Geospatial Tasks.* SIGSPATIAL 2025. [arXiv:2503.18129](https://arxiv.org/abs/2503.18129)
- *EO-Gym: A Multimodal, Interactive Environment for Earth Observation Agents.* [arXiv:2605.01250](https://arxiv.org/abs/2605.01250)
- Shabbir et al. *ThinkGeo: Evaluating Tool-Augmented Agents for Remote Sensing Tasks.* [arXiv:2505.23752](https://arxiv.org/abs/2505.23752)
- Kao et al. *Towards LLM Agents for Earth Observation* (UnivEARTH). [arXiv:2504.12110](https://arxiv.org/abs/2504.12110)
- Hou et al. *Can Large Language Models Generate Geospatial Code?* (GeoCode-Bench). [arXiv:2410.09738](https://arxiv.org/abs/2410.09738)
- Dihan et al. *MapEval: A Map-Based Evaluation of Geo-Spatial Reasoning in Foundation Models.* ICML 2025. [arXiv:2501.00316](https://arxiv.org/abs/2501.00316)
- Yao et al. *ReAct: Synergizing Reasoning and Acting in Language Models.* [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
- Chen et al. *Evaluating Large Language Models Trained on Code* (Pass@k). [arXiv:2107.03374](https://arxiv.org/abs/2107.03374)
- Atrio et al. *EVE: A Domain-Specific LLM Framework for Earth Intelligence.* ACL 2026.
- [Inspect AI](https://inspect.aisi.org.uk/), [LangGraph](https://github.com/langchain-ai/langgraph), [Model Context Protocol](https://modelcontextprotocol.io), [agentevals](https://github.com/langchain-ai/agentevals)